In [2]:
!pip install onnx

   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ------------ --------------------------- 5.2/16.5 MB 79.1 MB/s eta 0:00:01
   -------------------- ------------------- 8.7/16.5 MB 28.2 MB/s eta 0:00:01
   ---------------------------------------- 16.5/16.5 MB 33.4 MB/s  0:00:00

   ---------------------------------------- 0/3 [protobuf]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ---

In [1]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
import torch


# 네 학습된 ST 모델 폴더
st_path = "./sbert_specup_model"


st = SentenceTransformer(st_path)
# ST 내부 transformer 이름 가져오기 (보통 0번 모듈)
hf_model_name = st._first_module().auto_model.name_or_path


tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
model = AutoModel.from_pretrained(hf_model_name)
model.eval()


dummy = tokenizer("hello world", return_tensors="pt")


torch.onnx.export(
  model,
  (dummy["input_ids"], dummy["attention_mask"]),
  "encoder.onnx",
  input_names=["input_ids", "attention_mask"],
  output_names=["last_hidden_state"],
  dynamic_axes={
    "input_ids": {0: "batch", 1: "seq"},
    "attention_mask": {0: "batch", 1: "seq"},
    "last_hidden_state": {0: "batch", 1: "seq"}
  },
  opset_version=17
)

##
# 토크나이저도 같이 저장해두면 자바에서 동일 vocab 사용 가능
tokenizer.save_pretrained("./tokenizer")
print("export done: encoder.onnx + tokenizer/")


c:\Users\KOSMO\miniconda3\envs\myvenv2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The tokenizer you are loading from './sbert_specup_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from './sbert_specup_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
C:\Users\KOSMO\AppData\Local\Temp\ipykernel_14720\2195304612

export done: encoder.onnx + tokenizer/
